# Sprint 4 Experiment — Gemini 3.1 Flash-Lite × All Datasets × [PROMPT]

Set `PROMPT` in Cell 2, then **Run All**.

| Setting | Value |
|---------|-------|
| Model | gemini-3.1-flash-lite (fixed) |
| Questions | all_questions_combined.csv (185 rows — Sprint 2 + Sprint 3 failures) |
| Prompt | ← set in Cell 2 |
| Chunk size | 3000 (author default — fixed) |
| Chunk overlap | 300 (author default — fixed) |
| Top-K | 5 (author default — fixed) |
| Temperature | 0.0 (professor requirement — fixed) |
| Embedding | all-MiniLM-L6-v2 (local, free) |

## Cell 1 — Setup paths

In [ ]:
import sys, os

project_root = '/Users/I772947/personal work/LLM Benchmark Team Project/LLM_Benchmark_Team_Project_2026'
sprint4_root = os.path.join(project_root, 'Sprint 4')
sprint3_uda  = os.path.join(project_root, 'Sprint 3', 'UDA-Benchmark')

for p in [sprint4_root, sprint3_uda]:
    if p not in sys.path:
        sys.path.insert(0, p)

os.chdir(sprint3_uda)  # UDA preprocess uses relative paths from this root
print(f'project_root : {project_root}')
print(f'sprint4_root : {sprint4_root}')
print(f'sprint3_uda  : {sprint3_uda}')
print(f'cwd          : {os.getcwd()}')

if not os.path.isdir(sprint3_uda):
    raise FileNotFoundError(f'sprint3_uda not found: {sprint3_uda}')

## Cell 2 — ★ CONFIGURE THIS ★

In [ ]:
import pandas as pd

# ── Only change PROMPT ────────────────────────────────────────────────────────
MODEL_KEY = 'gemini-3.1-flash-lite'   # fixed — do not change
PROMPT    = 'simple'                   # 'simple' (zero-shot) | 'cot' (chain-of-thought)
# ─────────────────────────────────────────────────────────────────────────────

QUESTIONS_CSV = os.path.join(sprint4_root, 'benchmark/questions/all_questions_combined.csv')
OUTPUT_DIR    = os.path.join(sprint4_root, f'experiments/{MODEL_KEY}/results')

# PDF directory lookup — maps dataset name → folder containing PDFs
UDA_PDF_BASE = os.path.join(sprint3_uda, 'dataset/src_doc_files_example')
PDF_DIRS = {
    'music_structured': os.path.join(UDA_PDF_BASE, 'music_docs'),
    'tathybrid':        os.path.join(UDA_PDF_BASE, 'tat_docs'),
    'finhybrid':        os.path.join(UDA_PDF_BASE, 'fin_docs'),
    'nqtext':           os.path.join(UDA_PDF_BASE, 'wiki_nq_docs/pdfs'),
    'fetatab':          os.path.join(UDA_PDF_BASE, 'wiki_feta_docs/pdfs'),
    'papertab':         os.path.join(UDA_PDF_BASE, 'paper_docs'),
    'papertext':        os.path.join(UDA_PDF_BASE, 'paper_docs'),
}

# Load and preview
df_all = pd.read_csv(QUESTIONS_CSV)
print(f'Total questions: {len(df_all)}')
print(df_all.groupby(['dataset','sprint']).size().reset_index(name='count').to_string(index=False))
print()

# Verify all PDFs exist
print('PDF availability check:')
missing_pdfs = []
for _, row in df_all.iterrows():
    pdf_dir = PDF_DIRS.get(row['dataset'], '')
    pdf_path = os.path.join(pdf_dir, row['doc_name'] + '.pdf')
    if not os.path.exists(pdf_path):
        missing_pdfs.append(f"{row['dataset']}/{row['doc_name']}")
missing_pdfs = list(dict.fromkeys(missing_pdfs))  # deduplicate
if missing_pdfs:
    print(f'  WARNING — {len(missing_pdfs)} PDFs not found:')
    for p in missing_pdfs: print(f'    {p}')
else:
    print(f'  All PDFs found.')

## Cell 3 — Run benchmark (all datasets)

In [ ]:
from framework.rag_runner import RAGRunner
import pandas as pd
from datetime import datetime

os.makedirs(OUTPUT_DIR, exist_ok=True)
all_results = []

for dataset, group_df in df_all.groupby('dataset'):
    print(f'\n{'='*60}')
    print(f'Dataset: {dataset}  ({len(group_df)} questions)')
    print(f'{'='*60}')

    pdf_dir = PDF_DIRS.get(dataset, '')
    if not pdf_dir or not os.path.isdir(pdf_dir):
        print(f'  SKIP — PDF directory not found: {pdf_dir}')
        continue

    # Save a per-dataset temp CSV so RAGRunner.run() can consume it
    tmp_csv = os.path.join(OUTPUT_DIR, f'_tmp_{dataset}.csv')
    group_df.to_csv(tmp_csv, index=False)

    # music_structured has no UDA eval — use a neutral dataset key for the runner
    runner_dataset = dataset if dataset != 'music_structured' else 'nqtext'

    runner = RAGRunner(model_key=MODEL_KEY, dataset=runner_dataset, prompt=PROMPT)
    results_df = runner.run(
        questions_csv=tmp_csv,
        pdf_dir=pdf_dir,
        doc_col='doc_name',
        output_dir=OUTPUT_DIR,
    )
    results_df['dataset_actual'] = dataset  # preserve real dataset label
    all_results.append(results_df)
    os.remove(tmp_csv)

# Combine all results
results_combined = pd.concat(all_results, ignore_index=True)
ts = datetime.now().strftime('%Y%m%d_%H%M%S')
final_path = os.path.join(OUTPUT_DIR, f'all_datasets_{MODEL_KEY}_{PROMPT}_{ts}.csv')
results_combined.to_csv(final_path, index=False)
print(f'\nAll results saved → {final_path}')
print(f'Total rows: {len(results_combined)}')

## Cell 4 — Score Sprint 3 datasets (F1 / EM / Numeracy F1)

In [ ]:
from framework.scorer import score_results
from framework.config import DATASET_METRICS

# Score each Sprint 3 dataset separately (Sprint 2 music_structured has no UDA eval)
sprint3_datasets = [d for d in results_combined['dataset_actual'].unique() if d != 'music_structured']

scores = {}
for dataset in sprint3_datasets:
    subset = results_combined[results_combined['dataset_actual'] == dataset].copy()
    tmp_path = os.path.join(OUTPUT_DIR, f'_score_tmp_{dataset}.csv')
    subset.to_csv(tmp_path, index=False)
    info = score_results(tmp_path, dataset=dataset, save=False)
    scores[dataset] = info
    os.remove(tmp_path)

print('\n=== SCORE SUMMARY ===')
for ds, info in scores.items():
    print(f'  {ds}: metric={info["metric"]}, empty={info["empty_count"]}/{info["total"]}')

## Cell 5 — Quick failure analysis

In [ ]:
results_combined['is_empty'] = results_combined['response'].fillna('').str.strip() == ''
results_combined['resp_len'] = results_combined['response'].fillna('').str.len()

print('=== Empty responses by dataset ===')
summary = results_combined.groupby('dataset_actual').agg(
    total=('question_id','count'),
    empty=('is_empty','sum'),
    avg_resp_len=('resp_len','mean'),
).reset_index()
summary['empty_pct'] = (summary['empty'] / summary['total'] * 100).round(1)
print(summary.to_string(index=False))

print('\n=== Sample empty questions ===')
empty_rows = results_combined[results_combined['is_empty']]
for _, row in empty_rows.head(5).iterrows():
    print(f"  [{row.get('question_id','?')}] [{row['dataset_actual']}] {row['question'][:80]}")